# Context length in a small character-level transformer — Week 5 baseline gate

**Research question:** with the number of training characters held fixed, does increasing context length from 32 to 128 to 256 lower test bits-per-character (BPC) for a 4-layer character transformer on TinyShakespeare, by more than the run-to-run variation across 5 seeds?

This notebook is the **end-to-end pipeline** for the baseline gate. It covers:

| Gate evidence | Section |
|---|---|
| Data loading and preprocessing | 2 |
| Completed reference-condition run (n-gram) | 3 |
| Model construction | 4 |
| Working training loop | 5 |
| Metric calculation (fixed-target BPC) | 5 |
| Completed reference-condition run (context-32 transformer) | 6 |
| Saved configuration and results | every run writes to `results/` and `runs/` |

**Before running:** in Colab, use *Runtime → Change runtime type → T4 GPU*. The model and training code are adapted from Karpathy's nanoGPT (MIT licence).

**The test set is not touched in this notebook.** It is evaluated once, after all runs and decisions are frozen (section 8, switched off by default).

## 1. Setup

In [ ]:
# Optional (Colab): save everything to Google Drive so files survive a disconnect.
import os
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/ctxlen_project", exist_ok=True)
    os.chdir("/content/drive/MyDrive")

In [ ]:
import os, json, math, time, random, hashlib, platform, urllib.request, csv
from collections import Counter, defaultdict
from contextlib import nullcontext
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

BASE_DIR = "ctxlen_project"
DATA_DIR = os.path.join(BASE_DIR, "data")
RUNS_DIR = os.path.join(BASE_DIR, "runs")          # checkpoints, one folder per run
RESULTS_DIR = os.path.join(BASE_DIR, "results")    # small JSON/CSV/PNG results
for d in (DATA_DIR, RUNS_DIR, RESULTS_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"     # fp16 mixed precision on GPU (a T4 has no bf16)

# Set True only to check the pipeline quickly (tiny runs). Must be False for real results.
# Quick-test rows are flagged in the results CSV; delete the ctxlen_project folder before real runs.
QUICK_TEST = False

MAX_CTX = 256    # longest context in the study; evaluation targets start at this index in every split
SEEDS = [0, 1, 2, 3, 4]
CONTEXT_LENGTHS = [32, 128, 256]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()

env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "numpy": np.__version__,
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "cuda": torch.version.cuda,
}
with open(os.path.join(RESULTS_DIR, "
                       
                       "), "w") as f:
    json.dump(env, f, indent=2)
print(env)

## 2. Data loading and preprocessing

- Source: `input.txt` from Karpathy's char-rnn repository (TinyShakespeare, public-domain text).
- **Contiguous** 80/10/10 split, because a random split would leak neighbouring text between sets.
- Vocabulary built from the **training split only**. Any character in validation/test that never appears in training is mapped to an `<unk>` symbol and reported.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw_path = os.path.join(DATA_DIR, "input.txt")
if not os.path.exists(raw_path):
    urllib.request.urlretrieve(DATA_URL, raw_path)

with open(raw_path, encoding="utf-8") as f:
    text = f.read()

n = len(text)
n_train, n_val = int(0.8 * n), int(0.1 * n)
train_text = text[:n_train]
val_text = text[n_train:n_train + n_val]
test_text = text[n_train + n_val:]

UNK = "\x00"                                  # placeholder for characters unseen in training
train_chars = sorted(set(train_text))
assert UNK not in train_chars
itos = train_chars + [UNK]
stoi = {c: i for i, c in enumerate(itos)}
VOCAB_SIZE = len(itos)

unseen = {name: sorted(set(t) - set(train_chars)) for name, t in [("val", val_text), ("test", test_text)]}

def map_unseen(t):
    return "".join(c if c in stoi else UNK for c in t)

val_text = map_unseen(val_text)
test_text = map_unseen(test_text)

def encode(t):
    return torch.tensor([stoi[c] for c in t], dtype=torch.long)

train_data, val_data, test_data = encode(train_text), encode(val_text), encode(test_text)

data_meta = {
    "source_url": DATA_URL,
    "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
    "total_chars": n,
    "split_sizes": {"train": len(train_text), "val": len(val_text), "test": len(test_text)},
    "split": "contiguous 80/10/10",
    "vocab_size_including_unk": VOCAB_SIZE,
    "unseen_chars_mapped_to_unk": {k: [repr(c) for c in v] for k, v in unseen.items()},
    "eval_targets_start_index": MAX_CTX,
}
with open(os.path.join(RESULTS_DIR, "data_meta.json"), "w") as f:
    json.dump(data_meta, f, indent=2)
print(json.dumps(data_meta, indent=2))
print("\nStart of the training text:\n", train_text[:200])

## 3. Reference condition 1: character n-gram model

An **interpolated absolute-discounting** n-gram (a simplified relative of Kneser–Ney). For a history *h* and next character *c*:

P_n(c | h) = max(count(h, c) − d, 0) / count(h) + d · types(h) / count(h) · P_{n−1}(c | shorter h)

where types(h) is the number of different characters seen after *h*. Unseen histories back off fully to the shorter one; the unigram level uses add-one smoothing so no character ever gets probability zero.

**Tuned on validation:** order 2–8 and discount *d* ∈ {0.5, 0.75, 0.9}. It is deterministic, so it needs one run. It is scored on exactly the same target characters as the transformer (every character from index 256 onward).

In [ ]:
class InterpolatedNgram:
    def __init__(self, train_text, max_order, vocab):
        self.max_order = max_order
        self.V = len(vocab)
        self.N = len(train_text)
        self.unigram = Counter(train_text)
        self.ngram, self.ctx_total, self.ctx_types = {}, {}, {}
        for n in range(2, max_order + 1):
            grams = Counter(train_text[i:i + n] for i in range(len(train_text) - n + 1))
            tot, types = defaultdict(int), defaultdict(int)
            for g, c in grams.items():
                tot[g[:-1]] += c
                types[g[:-1]] += 1
            self.ngram[n], self.ctx_total[n], self.ctx_types[n] = grams, dict(tot), dict(types)

    def probs_all_orders(self, history, ch, d):
        # Probability of ch under every order 1..max_order. history needs >= max_order-1 characters.
        p = (self.unigram.get(ch, 0) + 1) / (self.N + self.V)
        out = [p]
        for n in range(2, self.max_order + 1):
            h = history[len(history) - (n - 1):]
            tot = self.ctx_total[n].get(h)
            if tot:
                c = self.ngram[n].get(h + ch, 0)
                p = max(c - d, 0) / tot + d * self.ctx_types[n][h] / tot * p
            out.append(p)
        return out

def ngram_bpc_all_orders(model, split_text, d, start=MAX_CTX):
    k = model.max_order
    total_bits = np.zeros(k)
    for t in range(start, len(split_text)):
        total_bits -= np.log2(model.probs_all_orders(split_text[t - (k - 1):t], split_text[t], d))
    return total_bits / (len(split_text) - start)

t0 = time.time()
ngram = InterpolatedNgram(train_text, max_order=8, vocab=itos)
print(f"n-gram counts built in {time.time() - t0:.1f}s")

# Sanity check: probabilities over the vocabulary must sum to 1 for every order.
hist = val_text[1000:1007]
for d in (0.5, 0.9):
    sums = np.sum([ngram.probs_all_orders(hist, c, d) for c in itos], axis=0)
    assert np.allclose(sums, 1.0), sums
print("Probabilities sum to 1 for every order: OK")

In [ ]:
D_GRID = [0.5, 0.75, 0.9]
grid = []
for d in D_GRID:
    bpcs = ngram_bpc_all_orders(ngram, val_text, d)
    for order, b in enumerate(bpcs, start=1):
        grid.append({"order": order, "discount": d, "val_bpc": float(b)})
    print(f"d={d}: " + "  ".join(f"n={o}:{b:.3f}" for o, b in enumerate(bpcs, start=1)))

best_ngram = min((g for g in grid if g["order"] >= 2), key=lambda g: g["val_bpc"])
ngram_results = {"condition": "ngram", "selection": "lowest validation BPC", "best": best_ngram,
                 "grid": grid, "test_bpc": None}
with open(os.path.join(RESULTS_DIR, "ngram_results.json"), "w") as f:
    json.dump(ngram_results, f, indent=2)
print("\nSelected n-gram (on validation):", best_ngram)

## 4. Model construction

A small GPT-style decoder-only transformer, adapted from nanoGPT: pre-LayerNorm blocks, causal self-attention, GELU MLP, **learned position embeddings**, tied input/output embeddings.

One detail matters for the controlled comparison: the position table is registered **last** and all weights are initialised in one pass after seeding. So with the same seed, every shared weight starts identical across context lengths; only the position table differs (it has a different number of rows). This is checked below.

In [ ]:
@dataclass
class Config:
    context_len: int = 32
    seed: int = 0
    # model (held fixed across conditions)
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 192
    dropout: float = 0.1
    # training budget (held fixed across conditions)
    chars_per_step: int = 16384
    max_steps: int = 3000
    lr: float = 1e-3
    min_lr: float = 1e-4
    warmup_steps: int = 100
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.99
    grad_clip: float = 1.0
    # validation-based checkpoint selection
    eval_interval: int = 250
    eval_batches: int = 20
    eval_seed: int = 1234     # the same validation batches are drawn every time

    @property
    def batch_size(self):
        assert self.chars_per_step % self.context_len == 0
        return self.chars_per_step // self.context_len

    @property
    def run_name(self):
        return f"ctx{self.context_len}_seed{self.seed}"


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.proj.is_residual_proj = True
        self.attn_dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, heads, T, head_size)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        # causal mask: each position attends only to itself and earlier positions
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.attn_dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.proj.is_residual_proj = True
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.proj(F.gelu(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class CharGPT(nn.Module):
    def __init__(self, cfg, vocab_size):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(vocab_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight        # weight tying, as in nanoGPT
        self.pos_emb = nn.Embedding(cfg.context_len, cfg.n_embd)   # registered LAST on purpose
        torch.manual_seed(cfg.seed)
        self.apply(self._init_weights)                # one pass, in registration order

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02 / math.sqrt(2 * self.cfg.n_layer) if getattr(m, "is_residual_proj", False) else 0.02
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, last_only=False):
        B, T = idx.shape
        assert T <= self.cfg.context_len
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if last_only:                                  # used by the fixed-target evaluation
            return self.head(x[:, -1, :])
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def param_counts(self):
        total = sum(p.numel() for p in self.parameters())
        return {"total": total, "position_table": self.pos_emb.weight.numel(),
                "excluding_position_table": total - self.pos_emb.weight.numel()}

    @torch.no_grad()
    def generate(self, idx, n_new, temperature=0.8):
        self.eval()                                    # no dropout while sampling
        for _ in range(n_new):
            logits = self.head(self.ln_f(self._trunk(idx[:, -self.cfg.context_len:])))[:, -1, :]
            probs = F.softmax(logits / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

    def _trunk(self, idx):
        pos = torch.arange(idx.size(1), device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        return x


# Checks: parameter counts per condition, and identical starting weights across context lengths.
for L in CONTEXT_LENGTHS:
    print(f"context {L}: batch {Config(context_len=L).batch_size}, params {CharGPT(Config(context_len=L), VOCAB_SIZE).param_counts()}")

m32 = CharGPT(Config(context_len=32, seed=0), VOCAB_SIZE)
m256 = CharGPT(Config(context_len=256, seed=0), VOCAB_SIZE)
sd32, sd256 = m32.state_dict(), m256.state_dict()
assert all(torch.equal(sd32[k], sd256[k]) for k in sd32 if k != "pos_emb.weight")
print("Same seed -> identical starting weights across context lengths (except position table): OK")
del m32, m256, sd32, sd256

## 5. Training loop and metric

- **Budget:** 16,384 characters per step for every condition (batch 512 / 128 / 64), same maximum steps.
- **Optimiser:** AdamW (weight decay on weight matrices only), linear warmup then cosine decay, gradient clipping 1.0, fp16 mixed precision on GPU.
- **Checkpoint selection:** lowest validation loss, estimated on a fixed set of validation batches (same characters-per-batch for every condition).
- **Resumable:** the full training state is saved at every evaluation, so a disconnected run continues where it stopped.
- **Metric:** fixed-target BPC. Every model predicts the same target characters (index 256 onward), each given the preceding L characters; only the final position of each window is scored.

In [ ]:
def get_batch(data, L, B, gen):
    ix = torch.randint(len(data) - L - 1, (B,), generator=gen)
    offsets = ix[:, None] + torch.arange(L)
    return data[offsets].to(DEVICE), data[offsets + 1].to(DEVICE)

def get_lr(step, cfg):
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / cfg.warmup_steps
    progress = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    return cfg.min_lr + 0.5 * (cfg.lr - cfg.min_lr) * (1 + math.cos(math.pi * min(progress, 1.0)))

def make_optimizer(model, cfg):
    decay = [p for p in model.parameters() if p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.dim() < 2]
    return torch.optim.AdamW([{"params": decay, "weight_decay": cfg.weight_decay},
                              {"params": no_decay, "weight_decay": 0.0}],
                             lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))

def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

@torch.no_grad()
def estimate_val_loss(model, cfg):
    model.eval()
    gen = torch.Generator().manual_seed(cfg.eval_seed)
    losses = []
    for _ in range(cfg.eval_batches):
        x, y = get_batch(val_data, cfg.context_len, cfg.batch_size, gen)
        with autocast_ctx():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

@torch.no_grad()
def fixed_target_bpc(model, data, L, start=MAX_CTX, windows_per_batch=512, max_targets=None):
    # Same targets for every model: data[start:], each predicted from the preceding L characters.
    model.eval()
    targets = torch.arange(start, len(data))
    if max_targets is not None:
        targets = targets[:max_targets]
    ar = torch.arange(L)
    total_nll = 0.0
    for i in range(0, len(targets), windows_per_batch):
        t = targets[i:i + windows_per_batch]
        x = data[(t[:, None] - L) + ar].to(DEVICE)
        y = data[t].to(DEVICE)
        with autocast_ctx():
            logits = model(x, last_only=True)
        total_nll += F.cross_entropy(logits.float(), y, reduction="sum").item()
    model.train()
    return total_nll / len(targets) / math.log(2)

In [ ]:
RESULTS_CSV = os.path.join(RESULTS_DIR, "transformer_runs.csv")

def train_run(cfg, resume=True):
    run_dir = os.path.join(RUNS_DIR, cfg.run_name)
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(asdict(cfg) | {"batch_size": cfg.batch_size}, f, indent=2)

    set_seed(cfg.seed)
    model = CharGPT(cfg, VOCAB_SIZE).to(DEVICE)
    opt = make_optimizer(model, cfg)
    scaler = make_scaler()
    gen = torch.Generator().manual_seed(cfg.seed)      # controls which training batches are drawn
    start_step, history = 0, []
    best_val, best_step, elapsed = float("inf"), -1, 0.0

    last_path, best_path = os.path.join(run_dir, "last.pt"), os.path.join(run_dir, "best.pt")
    if resume and os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        gen.set_state(ck["gen_state"]); torch.set_rng_state(ck["torch_rng"])
        if DEVICE == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(ck["cuda_rng"])
        start_step, history = ck["step"], ck["history"]
        best_val, best_step, elapsed = ck["best_val"], ck["best_step"], ck["elapsed"]
        print(f"Resumed {cfg.run_name} at step {start_step}")
        if start_step >= cfg.max_steps:
            print("Training already complete.")

    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    model.train()
    t0, recent = time.time(), []
    for step in range(start_step, cfg.max_steps):
        for g in opt.param_groups:
            g["lr"] = get_lr(step, cfg)
        x, y = get_batch(train_data, cfg.context_len, cfg.batch_size, gen)
        with autocast_ctx():
            _, loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
        recent.append(loss.item())

        is_last = step == cfg.max_steps - 1
        if (step + 1) % cfg.eval_interval == 0 or is_last:
            val_loss = estimate_val_loss(model, cfg)
            history.append({"step": step + 1, "train_loss": float(np.mean(recent)), "val_loss": val_loss})
            recent = []
            if val_loss < best_val:
                best_val, best_step = val_loss, step + 1
                torch.save({"model": model.state_dict(), "step": best_step}, best_path)
            run_time = elapsed + time.time() - t0
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "gen_state": gen.get_state(), "torch_rng": torch.get_rng_state(),
                        "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None,
                        "step": step + 1, "history": history, "best_val": best_val,
                        "best_step": best_step, "elapsed": run_time}, last_path)
            print(f"step {step + 1:5d} | train {history[-1]['train_loss']:.4f} | val {val_loss:.4f} "
                  f"| best {best_val:.4f} @ {best_step} | {run_time / 60:.1f} min")
    elapsed += time.time() - t0

    # Evaluate the selected (best-validation) checkpoint with the fixed-target metric.
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=False)["model"])
    max_targets = 2000 if QUICK_TEST else None
    val_bpc = fixed_target_bpc(model, val_data, cfg.context_len, max_targets=max_targets)

    result = {"run": cfg.run_name, "context_len": cfg.context_len, "seed": cfg.seed,
              "batch_size": cfg.batch_size, "max_steps": cfg.max_steps, "best_step": best_step,
              "best_val_loss_estimate_nats": best_val, "val_bpc_fixed_target": val_bpc,
              "test_bpc_fixed_target": None,
              "train_minutes": elapsed / 60,
              "peak_gpu_mem_mb": torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else None,
              **{f"params_{k}": v for k, v in model.param_counts().items()},
              "quick_test": QUICK_TEST}
    with open(os.path.join(run_dir, "result.json"), "w") as f:
        json.dump(result | {"history": history}, f, indent=2)
    write_header = not os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(result))
        if write_header:
            w.writeheader()
        w.writerow(result)
    print(f"\n{cfg.run_name}: val BPC (fixed targets) = {val_bpc:.4f}, best step {best_step}, "
          f"{elapsed / 60:.1f} min")
    return model, result, history

## 6. Reference condition 2: context-32 transformer (seed 0)

This is the gate's completed transformer run. Learning rate 1e-3 is a starting value; the learning-rate pilot at context 128 (on validation) comes next and may change it before the main runs.

In [ ]:
cfg = Config(context_len=32, seed=0)
if QUICK_TEST:
    cfg.max_steps, cfg.eval_interval, cfg.eval_batches, cfg.warmup_steps = 60, 20, 2, 10
print(cfg, "| batch size:", cfg.batch_size)
model, result, history = train_run(cfg)

In [ ]:
steps = [h["step"] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(steps, [h["train_loss"] for h in history], label="train (mean since last eval)")
plt.plot(steps, [h["val_loss"] for h in history], label="validation (fixed batches)")
plt.axvline(result["best_step"], ls="--", c="grey", label=f"selected checkpoint (step {result['best_step']})")
plt.xlabel("training step"); plt.ylabel("cross-entropy (nats per character)")
plt.title(f"{cfg.run_name}: training curve"); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"curve_{cfg.run_name}.png"), dpi=150)
plt.show()

# A short sample from the selected checkpoint, as a qualitative sanity check only.
start = torch.tensor([[stoi["\n"]]], device=DEVICE)
sample = "".join(itos[i] for i in model.generate(start, 300)[0].tolist())
with open(os.path.join(RESULTS_DIR, f"sample_{cfg.run_name}.txt"), "w", encoding="utf-8") as f:
    f.write(sample)
print(sample)

## 7. Baseline results so far (validation only)

In [ ]:
import pandas as pd
runs = pd.read_csv(RESULTS_CSV)
print("Transformer runs:")
print(runs[["run", "context_len", "seed", "batch_size", "best_step", "val_bpc_fixed_target",
            "train_minutes", "params_total", "quick_test"]].to_string(index=False))
print(f"\nN-gram (order {best_ngram['order']}, d={best_ngram['discount']}): "
      f"val BPC = {best_ngram['val_bpc']:.4f}")
print("Uniform guessing over the vocabulary would give", f"{math.log2(VOCAB_SIZE):.4f}", "BPC")

## 8. Later stages (switched off for the baseline gate)

**Main experiment:** set `RUN_FULL_EXPERIMENT = True` after the learning-rate pilot. Runs are resumable, so the cell can be re-run after a disconnect and finished runs are skipped.

**Test evaluation:** set `RUN_TEST = True` **once**, after all runs and decisions are frozen.

In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    for L in CONTEXT_LENGTHS:
        for seed in SEEDS:
            done = os.path.join(RUNS_DIR, f"ctx{L}_seed{seed}", "result.json")
            if os.path.exists(done):
                print("skip (done):", done); continue
            train_run(Config(context_len=L, seed=seed))

In [ ]:
RUN_TEST = False   # one-time final evaluation; do not use for any decision
if RUN_TEST:
    test_rows = []
    for L in CONTEXT_LENGTHS:
        for seed in SEEDS:
            c = Config(context_len=L, seed=seed)
            m = CharGPT(c, VOCAB_SIZE).to(DEVICE)
            m.load_state_dict(torch.load(os.path.join(RUNS_DIR, c.run_name, "best.pt"),
                                         map_location=DEVICE, weights_only=False)["model"])
            test_rows.append({"run": c.run_name, "context_len": L, "seed": seed,
                              "test_bpc": fixed_target_bpc(m, test_data, L)})
    k = best_ngram["order"]
    ngram_test = float(ngram_bpc_all_orders(ngram, test_text, best_ngram["discount"])[k - 1])
    pd.DataFrame(test_rows).to_csv(os.path.join(RESULTS_DIR, "test_results.csv"), index=False)
    with open(os.path.join(RESULTS_DIR, "ngram_test.json"), "w") as f:
        json.dump({"order": k, "discount": best_ngram["discount"], "test_bpc": ngram_test}, f, indent=2)
    print(pd.DataFrame(test_rows).groupby("context_len")["test_bpc"].agg(["mean", "std"]))
    print("n-gram test BPC:", ngram_test)

## Files saved (gate evidence)

- `results/environment.json` — hardware and library versions
- `results/data_meta.json` — data source, checksum, split sizes, vocabulary, unseen characters
- `results/ngram_results.json` — full n-gram validation grid and the selected setting
- `runs/ctx32_seed0/config.json` — exact configuration of the transformer run
- `runs/ctx32_seed0/result.json` — metrics plus the full training history
- `runs/ctx32_seed0/best.pt`, `last.pt` — selected checkpoint and resumable state
- `results/transformer_runs.csv` — one row per transformer run (becomes the results table)
- `results/curve_ctx32_seed0.png`, `results/sample_ctx32_seed0.txt` — figure and qualitative sample

**Next steps:** learning-rate pilot at context 128 on validation (check 32 and 256 as a diagnostic), check for overfitting to confirm dropout, then the 15 main runs.